# PS S6E6 — Ensemble v5

**What this does:** Combines OOF probabilities from v2-fix (LGBM), v3 (CatBoost), v4 (XGBoost)  
to find optimal ensemble weights, then applies those weights to test predictions.

**Setup:** Before running, attach the outputs of v2-fix, v3, and v4 notebooks as Kaggle input datasets.  
Each of those notebooks saves:
- `oof_proba_lgbm.csv` / `test_proba_lgbm.csv`
- `oof_proba_catboost.csv` / `test_proba_catboost.csv`
- `oof_proba_xgb.csv` / `test_proba_xgb.csv`

Update the `INPUT_PATHS` dict below to match where Kaggle mounts those files.

## 1. Imports & Paths

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score, classification_report

pd.set_option('display.float_format', '{:.5f}'.format)

In [ ]:
# Update these paths to match where Kaggle mounts each notebook's output
# e.g. /kaggle/input/<notebook-slug>/oof_proba_lgbm.csv
INPUT_PATHS = {
    'lgbm':     '/kaggle/input/lgbm-v2-fix',
    'catboost': '/kaggle/input/catboost-v3',
    'xgb':      '/kaggle/input/xgboost-v4',
}

BEST_LB = 0.96576  # update as scores come in

## 2. Load Train Labels + OOF/Test Probabilities

In [ ]:
train = pd.read_csv('/kaggle/input/datasets/ekowannanindome/stellar-dataset/train.csv', index_col='id')
test  = pd.read_csv('/kaggle/input/datasets/ekowannanindome/stellar-dataset/test.csv',  index_col='id')

le = LabelEncoder()
y  = le.fit_transform(train['class'])
true_labels = le.inverse_transform(y)
classes_ordered = ['GALAXY', 'QSO', 'STAR']

print('Classes:', dict(zip(le.classes_, le.transform(le.classes_))))

In [ ]:
def load_proba(base_path, name):
    oof  = pd.read_csv(f'{base_path}/oof_proba_{name}.csv').values
    test = pd.read_csv(f'{base_path}/test_proba_{name}.csv').values
    print(f'  {name}: oof={oof.shape}, test={test.shape}')
    return oof, test

print('Loading probabilities...')
oof_lgbm,  test_lgbm  = load_proba(INPUT_PATHS['lgbm'],     'lgbm')
oof_cat,   test_cat   = load_proba(INPUT_PATHS['catboost'],  'catboost')
oof_xgb,   test_xgb   = load_proba(INPUT_PATHS['xgb'],      'xgb')

## 3. Individual Model OOF Scores

In [ ]:
def oof_score(proba):
    return balanced_accuracy_score(true_labels, le.inverse_transform(proba.argmax(axis=1)))

score_lgbm = oof_score(oof_lgbm)
score_cat  = oof_score(oof_cat)
score_xgb  = oof_score(oof_xgb)

print(f'LGBM     OOF: {score_lgbm:.5f}')
print(f'CatBoost OOF: {score_cat:.5f}')
print(f'XGBoost  OOF: {score_xgb:.5f}')

## 4. Simple Average Ensemble

In [ ]:
oof_avg  = (oof_lgbm  + oof_cat  + oof_xgb)  / 3
test_avg = (test_lgbm + test_cat + test_xgb) / 3

score_avg = oof_score(oof_avg)
print(f'Simple average OOF: {score_avg:.5f}')
print(f'Best individual:    {max(score_lgbm, score_cat, score_xgb):.5f}')
print(f'Ensemble gain:      {score_avg - max(score_lgbm, score_cat, score_xgb):+.5f}')

## 5. Optimised Weight Search

Use Nelder-Mead to find weights `[w_lgbm, w_cat, w_xgb]` that maximise OOF balanced accuracy.  
Weights are constrained to sum to 1 and be non-negative.

In [ ]:
def neg_score(weights):
    w = np.array(weights)
    w = np.clip(w, 0, None)
    w /= w.sum()
    blend = w[0]*oof_lgbm + w[1]*oof_cat + w[2]*oof_xgb
    return -balanced_accuracy_score(true_labels, le.inverse_transform(blend.argmax(axis=1)))

# Initialise near equal weights
x0 = [1/3, 1/3, 1/3]
result = minimize(neg_score, x0, method='Nelder-Mead',
                  options={'maxiter': 10000, 'xatol': 1e-6, 'fatol': 1e-6})

best_w = np.clip(result.x, 0, None)
best_w /= best_w.sum()

score_opt = -result.fun
print(f'Optimised weights — LGBM: {best_w[0]:.3f} | CatBoost: {best_w[1]:.3f} | XGB: {best_w[2]:.3f}')
print(f'Optimised OOF:  {score_opt:.5f}')
print(f'Simple avg OOF: {score_avg:.5f}')
print(f'Gain from optimisation: {score_opt - score_avg:+.5f}')

## 6. Per-Class Report (Best Ensemble)

In [ ]:
# Use whichever scored better: simple avg or optimised
if score_opt > score_avg:
    final_oof_proba  = best_w[0]*oof_lgbm  + best_w[1]*oof_cat  + best_w[2]*oof_xgb
    final_test_proba = best_w[0]*test_lgbm + best_w[1]*test_cat + best_w[2]*test_xgb
    print(f'Using optimised weights: LGBM={best_w[0]:.3f}, Cat={best_w[1]:.3f}, XGB={best_w[2]:.3f}')
else:
    final_oof_proba  = oof_avg
    final_test_proba = test_avg
    print('Using simple average (optimised did not improve)')

final_oof_labels = le.inverse_transform(final_oof_proba.argmax(axis=1))
final_score = balanced_accuracy_score(true_labels, final_oof_labels)

print(f'\nFinal ensemble OOF: {final_score:.5f}  (best LB so far: {BEST_LB})')
print()
print(classification_report(true_labels, final_oof_labels, target_names=classes_ordered))

## 7. Generate Submission

In [ ]:
test_pred_labels = le.inverse_transform(final_test_proba.argmax(axis=1))
submission = pd.DataFrame({'id': test.index, 'class': test_pred_labels})
submission.to_csv('submission_ensemble_v5.csv', index=False)

print(f'Submission shape: {submission.shape}')
print(submission['class'].value_counts())

In [ ]:
from IPython.display import FileLink, display
display(FileLink('submission_ensemble_v5.csv'))

## 8. Results

| Model | OOF | LB | Notes |
|---|---|---|---|
| LGBM v2-fix | ... | ... | |
| CatBoost v3 | ... | ... | |
| XGBoost v4 | ... | ... | |
| **Ensemble v5** | **...** | **...** | LGBM w=... / Cat w=... / XGB w=... |

**Ensemble gain over best individual (OOF):**  
**STAR precision vs baseline (was 0.88):**